# Análisis Hidrometeorológico - Estación 34 CORNARE
Este notebook contiene los pasos 5 al 9 para el procesamiento de series de tiempo de niveles de agua o caudales, garantizando que se respete la causalidad temporal y no haya filtración de datos (data leakage).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

### 5. Missing Values Reales (Reindexación)
Al trabajar con series temporales reales, la ausencia de un dato a una hora específica es información. Reindexamos para forzar la aparición de `NaN` en los huecos temporales.

In [ ]:
# Supongamos que df es tu DataFrame y ya consultaste la API entre el 17 y 31 de agosto.
# df['fecha'] = pd.to_datetime(df['fecha'])
# df = df.set_index('fecha').sort_index()

# Ajusta '1h' (1 hora) o 'min' (1 minuto) según la frecuencia de envío de la estación 34
# df = df.asfreq('1h')

### 6. Tratamiento de Outliers
Combinamos el criterio físico (ej. el nivel no puede ser negativo) con el método estadístico (Rango Intercuartílico - IQR).

In [ ]:
def limpiar_outliers(df, columna):
    # 1. Límites físicos
    condicion_fisica = (df[columna] >= 0)
    
    # 2. Límites estadísticos (IQR)
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    condicion_iqr = (df[columna] >= limite_inf) & (df[columna] <= limite_sup)
    
    # Convertimos a NaN lo que no cumpla las condiciones (dejando los huecos temporales intactos)
    df.loc[~(condicion_fisica & condicion_iqr), columna] = np.nan
    return df

### 7. Split Cronológico
No usamos `train_test_split` aleatorio para no viajar en el tiempo. Cortamos el DataFrame secuencialmente.

In [ ]:
def split_cronologico(df, train_pct=0.70, val_pct=0.15):
    n = len(df)
    train_size = int(n * train_pct)
    val_size = int(n * val_pct)
    
    df_train = df.iloc[:train_size].copy()
    df_val = df.iloc[train_size:train_size+val_size].copy()
    df_test = df.iloc[train_size+val_size:].copy()
    
    return df_train, df_val, df_test

### 8. Normalización / Estandarización y 9. Estadística Descriptiva
Es vital ajustar el escalador **sólo con los datos de entrenamiento**.

In [ ]:
def estandarizar_y_describir(df_train, df_val, df_test, columna):
    scaler = StandardScaler()
    
    # Ajustar solo en Train (ignorando NaNs)
    scaler.fit(df_train[[columna]].dropna())
    
    # Transformar todos
    df_train[[columna]] = scaler.transform(df_train[[columna]])
    df_val[[columna]] = scaler.transform(df_val[[columna]])
    df_test[[columna]] = scaler.transform(df_test[[columna]])
    
    print("--- Estadísticas de Entrenamiento (Post-Normalización) ---")
    print(df_train[[columna]].describe())
    
    return df_train, df_val, df_test, scaler